## Configuration & File Paths

In [3]:
import os
from pathlib import Path

# Get notebook directory and construct data path
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
BASE_DATA_PATH = os.path.join(NOTEBOOK_DIR, "..", "DSA4264 Data")

# Input paths
COURSES_DIR = os.path.join(BASE_DATA_PATH, "NUS-SMU-SUTD Courses")
JOBS_PARQUET = os.path.join(BASE_DATA_PATH, "processed_jobs_dual_embeddings.parquet")

# Course CSV files
NUS_COURSES_CSV = os.path.join(COURSES_DIR, "nus_courses.csv")
SMU_COURSES_CSV = os.path.join(COURSES_DIR, "smu_courses.csv")
SUTD_COURSES_CSV = os.path.join(COURSES_DIR, "sutd_courses.csv")

# Output directories
OUTPUT_DIR = os.path.join(NOTEBOOK_DIR, "bertopic_visualizations_global")
# CHANGED: Output job matches to DSA4264 Data/Jobs by courses/ (overwrites for zoom_in_analysis)
UNIVERSITY_JOB_MATCHES_DIR = os.path.join(BASE_DATA_PATH, "Jobs by courses")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(UNIVERSITY_JOB_MATCHES_DIR, exist_ok=True)

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Data path: {BASE_DATA_PATH}")
print(f"Jobs parquet: {JOBS_PARQUET}")
print(f"Courses directory: {COURSES_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Job matches output: {UNIVERSITY_JOB_MATCHES_DIR}")

Notebook directory: /Users/teresaliau/dsa4264/notebooks
Data path: /Users/teresaliau/dsa4264/notebooks/../DSA4264 Data
Jobs parquet: /Users/teresaliau/dsa4264/notebooks/../DSA4264 Data/processed_jobs_dual_embeddings.parquet
Courses directory: /Users/teresaliau/dsa4264/notebooks/../DSA4264 Data/NUS-SMU-SUTD Courses
Output directory: /Users/teresaliau/dsa4264/notebooks/bertopic_visualizations_global
Job matches output: /Users/teresaliau/dsa4264/notebooks/../DSA4264 Data/Jobs by courses


In [4]:
# === ALL IMPORTS - RUN ONCE ===
import pandas as pd
import numpy as np
import json
import time
import requests
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance
from umap import UMAP
from hdbscan import HDBSCAN
from gensim import corpora
from gensim.models import CoherenceModel
import optuna

# Load embedding model ONCE (this takes 10-30 seconds)
print("Loading SentenceTransformer model (this may take 20-30 seconds)...")
embedding_model = SentenceTransformer('all-mpnet-base-v2')
print("✓ Model loaded successfully!")

Loading SentenceTransformer model (this may take 20-30 seconds)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Model loaded successfully!


In [8]:
# === PREREQUISITE GRAPH SETUP (NUS only) ===

# Loads and processes prerequisite relationships from dependent_mods_dedup.csv
# Removes % signs, parses module lists, filters to valid NUS modules
# Saves processed graph to: nus_prerequisite_graph.json

print("="*80)
print("PREREQUISITE GRAPH SETUP")
print("="*80)

print("\nEach university gets its own optimal weights:")
print("  • NUS: includes s_prereqs with optimal λ")
print("  • SMU/SUTD: no prereqs (only s_jobs, s_level, core_boost)")
print("\nRun cell 11 to compute and save weights to:")
print("  DSA4264 Data/ols_learned_weights_by_university.csv")
print("="*80)

# === PREREQUISITE GRAPH SETUP (NUS only) ===
import ast

nus_courses_df = pd.read_csv(NUS_COURSES_CSV)
nus_courses_df.columns = nus_courses_df.columns.str.strip().str.lower()
nus_module_codes = set(nus_courses_df['code'].dropna().astype(str).unique())

print(f"\n✓ {len(nus_module_codes)} NUS modules")

# Load prerequisite graph
prereq_csv_path = os.path.join(BASE_DATA_PATH, "dependent_mods_dedup.csv")
prereq_df = pd.read_csv(prereq_csv_path)
prereq_df['prereq_token'] = prereq_df['prereq_token'].str.replace('%', '', regex=False)
prereq_df['dependent_modules_dedup'] = prereq_df['dependent_modules_dedup'].apply(ast.literal_eval)

# Build unlocks_graph: prerequisite → [courses it unlocks]
unlocks_graph = {}
for _, row in prereq_df.iterrows():
    prereq = row['prereq_token']
    dependents = row['dependent_modules_dedup']
    
    if prereq in nus_module_codes:
        valid_dependents = [dep for dep in dependents if dep in nus_module_codes]
        if valid_dependents:
            unlocks_graph[prereq] = valid_dependents

print(f"✓ {len(unlocks_graph)} foundational NUS modules")

# Top foundational modules
top_foundational = sorted(unlocks_graph.items(), key=lambda x: len(x[1]), reverse=True)[:5]
print("\nTop 5 foundational modules:")
for code, unlocked in top_foundational:
    print(f"  {code}: unlocks {len(unlocked)} courses")

# Save processed graph for future use
prereq_graph_json = os.path.join(BASE_DATA_PATH, "nus_prerequisite_graph.json")
with open(prereq_graph_json, 'w') as f:
    json.dump(unlocks_graph, f, indent=2)
print(f"\n✓ Saved prerequisite graph to: {os.path.basename(prereq_graph_json)}")

PREREQUISITE GRAPH SETUP

Each university gets its own optimal weights:
  • NUS: includes s_prereqs with optimal λ
  • SMU/SUTD: no prereqs (only s_jobs, s_level, core_boost)

Run cell 11 to compute and save weights to:
  DSA4264 Data/ols_learned_weights_by_university.csv

✓ 381 NUS modules
✓ 81 foundational NUS modules

Top 5 foundational modules:
  ST2334: unlocks 18 courses
  ST2131: unlocks 17 courses
  MA2116: unlocks 17 courses
  MA2002: unlocks 14 courses
  MA2001: unlocks 12 courses

✓ Saved prerequisite graph to: nus_prerequisite_graph.json


In [6]:
def compute_prerequisite_propagation(degree_modules_df, unlocks_graph, lambda_decay):
    """
    Propagates relevance through prerequisite chains.
    
    R_total(c) = R_direct(c) + λ * Σ[R_direct(c') / depth(c')]
    
    Args:
        degree_modules_df: DataFrame with 'code' and 'R_direct' columns
        unlocks_graph: Dict mapping course_code -> list of courses it unlocks
        lambda_decay: Decay factor (0-1) for downstream relevance
    
    Returns:
        Dict mapping course_code -> R_total score
    """
    module_scores = degree_modules_df.set_index('code')['R_direct'].to_dict()
    R_total = {}
    
    for course_code in degree_modules_df['code']:
        direct_score = module_scores.get(course_code, 0.0)
        
        # BFS to find all descendants
        inherited_score = 0.0
        visited = set()
        queue = [(course_code, 0)]  # (code, depth)
        
        while queue:
            current, depth = queue.pop(0)
            if current in visited:
                continue
            visited.add(current)
            
            unlocked = unlocks_graph.get(current, [])
            for descendant in unlocked:
                if descendant in module_scores and depth > 0:
                    inherited_score += module_scores[descendant] / depth
                queue.append((descendant, depth + 1))
        
        R_total[course_code] = direct_score + lambda_decay * inherited_score
    
    return R_total

## Prerequisite Graph Setup

In [7]:
# The refined, lean fluff list
job_fluff = [
    "job", "description", "apply", "candidates", "interested", "hiring",
    "qualifications", "preferred", "required", "responsibilities",
    "equivalent", "company", "opportunities", "role", "title",
    "experience", "years", "year", "skills", "work", "working", "team",
    "teams", "knowledge", "ability", "strong", "using", "understanding",
    "including", "related", "key", "help", "plus"
]
all_stop_words = list(ENGLISH_STOP_WORDS) + job_fluff

os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_parquet(JOBS_PARQUET)
df = df.drop_duplicates(subset=['description']).copy()

# filter for full-time/permanent roles
allowed_types = {"Full Time", "Permanent", "Contract"}
def keeps_allowed_types(emp_types):
    try:
        return bool(set(emp_types) & allowed_types)
    except TypeError:
        return False
df = df[df['employmentTypes'].apply(keeps_allowed_types)].copy()

print(f"Total jobs to cluster: {len(df)}")

# REMOVED: embedding_model = SentenceTransformer('all-mpnet-base-v2')  # Now loaded in imports cell
all_job_embeddings = np.stack(df['embedding_mpnet'].values)


def tune_bertopic_params(docs, embeddings, vectorizer_model, n_trials=10):
    # Uses Optuna to find the mathematically best UMAP and HDBSCAN parameters
    cleaned_docs = [str(doc).lower().split() for doc in docs]
    id2word = corpora.Dictionary(cleaned_docs)

    def objective(trial):
        n_neighbors = trial.suggest_int("n_neighbors", 15, 50)
        n_components = trial.suggest_int("n_components", 3, 10)
        min_cluster_size = trial.suggest_int("min_cluster_size", 10, 50)

        temp_umap = UMAP(n_neighbors=n_neighbors, n_components=n_components, min_dist=0.0, metric='cosine', random_state=42)
        temp_hdbscan = HDBSCAN(min_cluster_size=min_cluster_size, min_samples=5, metric='euclidean', cluster_selection_method='eom')

        temp_topic_model = BERTopic(
            umap_model=temp_umap,
            hdbscan_model=temp_hdbscan,
            vectorizer_model=vectorizer_model,
            language="english",
            calculate_probabilities=False # Kept False during tuning to save massive memory
        )

        try:
            topics, _ = temp_topic_model.fit_transform(docs, embeddings=embeddings)

            # Penalize if it completely fails to find clusters
            if len(temp_topic_model.get_topic_info()) < 3:
                return 0.0

            topics_words = [[word for word, _ in temp_topic_model.get_topic(topic_id)] for topic_id in temp_topic_model.get_topics() if topic_id != -1]
            if not topics_words:
                return 0.0

            # Calculate Coherence
            coherence_model = CoherenceModel(topics=topics_words, texts=cleaned_docs, dictionary=id2word, coherence='c_v')
            coherence_score = coherence_model.get_coherence()

            # Penalize high outlier ratios
            outlier_ratio = topics.count(-1) / len(topics)

            return coherence_score * (1.0 - outlier_ratio)

        except Exception as e:
            return 0.0

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)

    print(f"   -> Best params found: {study.best_params} (Score: {study.best_value:.4f})")
    return study.best_params

def process_entire_dataset():
    print(f"\n{'='*60}")
    print("Processing Entire Global Dataset")
    print(f"{'='*60}")

    docs = df['description'].tolist()
    selected_embeddings = all_job_embeddings

    # NLP Sub-models
    vectorizer_model = CountVectorizer(stop_words=all_stop_words, ngram_range=(1, 3))
    ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)
    representation_model = [KeyBERTInspired(), MaximalMarginalRelevance(diversity=0.8)]

    # Find best parameters (reduced from 10 to 5 trials for speed)
    best_params = tune_bertopic_params(docs, selected_embeddings, vectorizer_model, n_trials=5)

    # Build the optimized sub-models
    umap_model = UMAP(
        n_neighbors=best_params['n_neighbors'],
        n_components=best_params['n_components'],
        min_dist=0.0,
        metric='cosine',
        random_state=42
    )

    hdbscan_model = HDBSCAN(
        min_cluster_size=best_params['min_cluster_size'],
        min_samples=5,
        metric='euclidean',
        cluster_selection_method='eom',
        prediction_data=True
    )

    # Assemble final BERTopic model (using pre-loaded embedding_model)
    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        ctfidf_model=ctfidf_model,
        representation_model=representation_model,
        calculate_probabilities=False, # Set to False for global run to prevent memory crash
        language="english"
    )

    # Fit the Model
    print("Fitting optimized global BERTopic model...")
    topics, _ = topic_model.fit_transform(docs, embeddings=selected_embeddings)

    topic_info = topic_model.get_topic_info()
    num_topics = len(topic_info) - 1 # Subtract 1 because Topic -1 is the outlier pile
    print(f"Discovered {num_topics} unique, optimized clusters across the entire market!\n")

    # Print a summary to Console
    for index, row in topic_info.iterrows():
        topic_id = row['Topic']
        count = row['Count']

        if topic_id == -1:
            print(f"  [Outliers] {count} jobs did not fit into a specific niche.")
            continue

        words = ", ".join(row['Representation'][:7])
        print(f"  Topic {topic_id:2d} ({count:4d} jobs) -> {words}")

    # Save Visualizations
    file_prefix = "global_market"

    if num_topics > 1:
        topic_model.visualize_topics().write_html(f"{OUTPUT_DIR}/{file_prefix}_distance_map.html")
        topic_model.visualize_hierarchy().write_html(f"{OUTPUT_DIR}/{file_prefix}_hierarchy.html")
        topic_model.visualize_barchart(top_n_topics=16).write_html(f"{OUTPUT_DIR}/{file_prefix}_barchart.html")
        topic_model.visualize_documents(docs, embeddings=selected_embeddings).write_html(f"{OUTPUT_DIR}/{file_prefix}_document_map.html")
        print(f"Saved global visualizations to /{OUTPUT_DIR}")
    else:
        print(f"Not enough topics found to generate visualizations.")

    # Save the fitted model
    topic_model.save(os.path.join(NOTEBOOK_DIR, "bertopic_global_model"), serialization="safetensors", save_ctfidf=True, save_embedding_model=embedding_model)

    # Annotate the jobs dataframe with their cluster assignments and save
    df_annotated = df.copy()
    df_annotated["topic_id"] = topics

    # Attach human-readable topic labels
    topic_label_map = {}
    for _, row in topic_model.get_topic_info().iterrows():
        tid = row["Topic"]
        if tid == -1:
            topic_label_map[tid] = "Outlier"
        else:
            topic_label_map[tid] = ", ".join(row["Representation"][:3])
    df_annotated["topic_label"] = df_annotated["topic_id"].map(topic_label_map)

    df_annotated.to_parquet(os.path.join(NOTEBOOK_DIR, "jobs_with_topics.parquet"), engine="pyarrow")
    print(f"Saved annotated jobs to 'jobs_with_topics.parquet'")
    print(f"Saved BERTopic model to 'bertopic_global_model/'")

    return topic_model

if __name__ == "__main__":
    topic_model = process_entire_dataset()
    print("Done, check your 'bertopic_visualizations_global' folder.")

Total jobs to cluster: 16686

Processing Entire Global Dataset
   -> Best params found: {'n_neighbors': 34, 'n_components': 9, 'min_cluster_size': 25} (Score: 0.0000)
Fitting optimized global BERTopic model...
Discovered 114 unique, optimized clusters across the entire market!

  [Outliers] 4376 jobs did not fit into a specific niche.
  Topic  0 ( 823 jobs) -> restaurant manager, restaurant operations, kitchen operations, kitchen staff, food preparation, head chef, culinary
  Topic  1 ( 690 jobs) -> accountancy, accountant, auditors, accounting finance, accounting finance field, accounts payable, accounting
  Topic  2 ( 630 jobs) -> classroom management, education teaching, childcare, teaching, childhood care education, teacher, teach
  Topic  3 ( 489 jobs) -> construction management, construction activities, engineering construction, construction works, civil engineering, contractor, construction drawings
  Topic  4 ( 488 jobs) -> marketing communications, marketing campaigns, marketi

In [9]:
# REMOVED: embedding_model = SentenceTransformer('all-mpnet-base-v2')  # Now loaded in imports cell
topic_model = BERTopic.load(os.path.join(NOTEBOOK_DIR, "bertopic_global_model"), embedding_model=embedding_model)

print("Extracting Topic Info...")
topic_info = topic_model.get_topic_info()

output_file = "global_topic_info.csv"
topic_info.to_csv(output_file, index=False)

print(f"Success! Saved all cluster definitions to '{output_file}'")

Extracting Topic Info...
Success! Saved all cluster definitions to 'global_topic_info.csv'


In [10]:
# NUS Module Embedding
OUTPUT_FILE = "10_nus_modules_embedded.parquet"
START_YEAR  = 2025

# REMOVED: embedding_model = SentenceTransformer("all-mpnet-base-v2")  # Now loaded in imports cell

# API fetcher
def fetch_module_data(mod_code, start_year):
    year = start_year
    while year >= 2015:
        url = f"https://api.nusmods.com/v2/{year}-{year+1}/modules/{mod_code.upper()}.json"
        r = requests.get(url)
        if r.status_code == 200:
            d = r.json()
            return {
                "code":          mod_code,
                "title":         d.get("title", ""),
                "description":   d.get("description", ""),
                "faculty":       d.get("faculty", ""),
                "department":    d.get("department", ""),
                "academic_year": f"{year}-{year+1}"
            }
        elif r.status_code == 404:
            year -= 1
        else:
            break
    return None


df = pd.read_csv(NUS_COURSES_CSV)
df.columns = df.columns.str.strip().str.lower()
df_unique = df.drop_duplicates(subset=["code"]).copy()

unique_modules = df_unique["code"].dropna().tolist()
print(f"Found {len(unique_modules)} unique modules across all majors.")

# Fetch module data from NUSMods API
print("Fetching module data from NUSMods API...")
all_records = []
for i, mod in enumerate(unique_modules, 1):
    if i % 50 == 0:
        print(f"  Progress: {i}/{len(unique_modules)}")
    record = fetch_module_data(mod, START_YEAR)
    if record:
        all_records.append(record)
    time.sleep(0.05)  # Rate limit

nus_df_unique = pd.DataFrame(all_records)
nus_df_unique = nus_df_unique[nus_df_unique["description"] != ""].reset_index(drop=True)

print(f"Found {len(nus_df_unique)} NUS modules with descriptions")

# Embed
nus_embeddings = embedding_model.encode(
    nus_df_unique["description"].tolist(),
    show_progress_bar=True
)
nus_df_unique["skill_embedding"] = list(nus_embeddings)

nus_output = os.path.join(COURSES_DIR, OUTPUT_FILE)
nus_df_unique.to_parquet(nus_output, engine="pyarrow")
print(f"Saved {len(nus_df_unique)} NUS modules to '{nus_output}'")

Found 381 unique modules across all majors.
Fetching module data from NUSMods API...
  Progress: 50/381
  Progress: 100/381
  Progress: 150/381
  Progress: 200/381
  Progress: 250/381
  Progress: 300/381
  Progress: 350/381
Found 369 NUS modules with descriptions


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Saved 369 NUS modules to '/Users/teresaliau/dsa4264/notebooks/../DSA4264 Data/NUS-SMU-SUTD Courses/10_nus_modules_embedded.parquet'


In [11]:
# SUTD Module Embedding
sutd_df = pd.read_csv(SUTD_COURSES_CSV)
sutd_df.columns = sutd_df.columns.str.strip().str.lower()
sutd_df_unique = sutd_df.drop_duplicates(subset=["code"]).copy()

# SUTD modules should have descriptions already in the CSV
sutd_df_unique["description"] = sutd_df_unique["description"].fillna("").str.strip()
sutd_df_unique = sutd_df_unique[sutd_df_unique["description"] != ""].reset_index(drop=True)

print(f"Found {len(sutd_df_unique)} SUTD modules with descriptions")

# Embed
sutd_embeddings = embedding_model.encode(
    sutd_df_unique["description"].tolist(),
    show_progress_bar=True
)
sutd_df_unique["skill_embedding"] = list(sutd_embeddings)

sutd_output = os.path.join(COURSES_DIR, "10_sutd_modules_embedded.parquet")
sutd_df_unique.to_parquet(sutd_output, engine="pyarrow")
print(f"Saved {len(sutd_df_unique)} SUTD modules to '{sutd_output}'")

Found 131 SUTD modules with descriptions


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Saved 131 SUTD modules to '/Users/teresaliau/dsa4264/notebooks/../DSA4264 Data/NUS-SMU-SUTD Courses/10_sutd_modules_embedded.parquet'


In [12]:
# SMU Module Embedding
smu_df = pd.read_csv(SMU_COURSES_CSV)
smu_df.columns = smu_df.columns.str.strip().str.lower()
smu_df_unique = smu_df.drop_duplicates(subset=["code"]).copy()

# SMU modules should have descriptions already in the CSV
smu_df_unique["description"] = smu_df_unique["description"].fillna("").str.strip()
smu_df_unique = smu_df_unique[smu_df_unique["description"] != ""].reset_index(drop=True)

print(f"Found {len(smu_df_unique)} SMU modules with descriptions")

# Embed
smu_embeddings = embedding_model.encode(
    smu_df_unique["description"].tolist(),
    show_progress_bar=True
)
smu_df_unique["skill_embedding"] = list(smu_embeddings)

smu_output = os.path.join(COURSES_DIR, "10_smu_modules_embedded.parquet")
smu_df_unique.to_parquet(smu_output, engine="pyarrow")
print(f"Saved {len(smu_df_unique)} SMU modules to '{smu_output}'")

Found 305 SMU modules with descriptions


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Saved 305 SMU modules to '/Users/teresaliau/dsa4264/notebooks/../DSA4264 Data/NUS-SMU-SUTD Courses/10_smu_modules_embedded.parquet'


In [13]:
# === STAGE 3 & 4: COMPUTE s_jobs AND s_prereqs FOR EACH DEGREE ===

import re
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

print("="*80)
print("UNIVERSITY-SPECIFIC OLS WITH LAMBDA GRID SEARCH")
print("="*80)

def extract_module_level(module_code):
    """Extract module level (1000, 2000, 3000, 4000) from code."""
    module_code = str(module_code)
    match = re.search(r'(\d)', module_code)
    if match:
        return int(match.group(1)) * 1000
    return 1000

def compute_s_jobs(modules_df, job_matches_df):
    """Direct job relevance: average cosine similarity to matched jobs."""
    s_jobs = {}
    
    # BUGFIX: Extract just the module code from various formats:
    # NUS: "DSA4262 – Sense-making Case..." → "DSA4262"
    # SMU: "7941 (elective)" → "7941"
    # SUTD: "50-055 (elective)" → "50-055"
    job_matches_df = job_matches_df.copy()
    
    # First remove " (elective)" or " (core)"
    job_matches_df['best_module_clean'] = job_matches_df['best_module'].astype(str).str.replace(r' \(.*\)', '', regex=True)
    
    # Then extract module code before " –" or " -" (NUS course titles)
    job_matches_df['best_module_clean'] = job_matches_df['best_module_clean'].str.split(r' [–-] ').str[0].str.strip()
    
    for module_code in modules_df['code']:
        module_code = str(module_code)
        module_jobs = job_matches_df[job_matches_df['best_module_clean'] == module_code]
        s_jobs[module_code] = module_jobs['max_similarity'].mean() if len(module_jobs) > 0 else 0.0
    
    return s_jobs

def compute_s_prereqs_with_lambda(modules_df, s_jobs, unlocks_graph, lambda_decay):
    """Prerequisite importance with BFS decay: R_total = R_direct + λ * Σ[R_direct(child) / depth]"""
    R_total = {}
    for course_code in modules_df['code']:
        course_code = str(course_code)
        direct_score = s_jobs.get(course_code, 0.0)
        
        # BFS to find all descendants with depth
        inherited_score = 0.0
        visited = set()
        queue = [(course_code, 0)]
        
        while queue:
            current, depth = queue.pop(0)
            if current in visited:
                continue
            visited.add(current)
            
            unlocked = unlocks_graph.get(current, [])
            for descendant in unlocked:
                if descendant in s_jobs and depth > 0:
                    inherited_score += s_jobs[descendant] / depth
                queue.append((descendant, depth + 1))
        
        R_total[course_code] = direct_score + lambda_decay * inherited_score
    
    return R_total

# Load GES employment data
ges_df = pd.read_csv(os.path.join(BASE_DATA_PATH, "graduate_employment_survey_2025.csv"))

all_ols_results = []
all_merged_dfs = []  # Collect all merged dataframes

for university in ['nus', 'smu', 'sutd']:
    print(f"\n{'='*80}")
    print(f"{university.upper()}")
    print(f"{'='*80}")
    
    if university == 'nus':
        course_csv = NUS_COURSES_CSV
        module_parquet = os.path.join(COURSES_DIR, "10_nus_modules_embedded.parquet")
    elif university == 'smu':
        course_csv = SMU_COURSES_CSV
        module_parquet = os.path.join(COURSES_DIR, "10_smu_modules_embedded.parquet")
    else:
        course_csv = SUTD_COURSES_CSV
        module_parquet = os.path.join(COURSES_DIR, "10_sutd_modules_embedded.parquet")
    
    try:
        modules_df = pd.read_parquet(module_parquet)
        csv_df = pd.read_csv(course_csv)
        csv_df.columns = csv_df.columns.str.strip().str.lower()
    except:
        print(f"⚠ Could not load data for {university.upper()}")
        continue
    
    # Grid search over lambda for NUS, no prereqs for SMU/SUTD
    lambda_values = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9] if university == 'nus' else [None]
    
    best_lambda = None
    best_r2 = -999
    best_model = None
    best_merged_df = None
    best_feature_names = None
    
    for lambda_test in lambda_values:
        if university == 'nus':
            print(f"  Testing λ = {lambda_test:.1f}...", end=" ")
        
        degree_features = []
        
        for course_name in csv_df["course"].dropna().unique():
            safe_course = str(course_name).replace(" ", "_").replace("/", "_").replace("\\", "_")
            match_file = os.path.join(UNIVERSITY_JOB_MATCHES_DIR, f"{university}_{safe_course}_matches.csv")
            
            if not os.path.exists(match_file):
                continue
            
            job_matches = pd.read_csv(match_file)
            degree_modules = csv_df[csv_df["course"] == course_name]["code"].dropna().astype(str).tolist()
            modules_df_degree = modules_df[modules_df["code"].astype(str).isin(degree_modules)].copy()
            
            if len(modules_df_degree) == 0:
                continue
            
            # Compute s_jobs
            s_jobs = compute_s_jobs(modules_df_degree, job_matches)
            
            # Compute s_prereqs (only for NUS)
            if university == 'nus':
                s_prereqs = compute_s_prereqs_with_lambda(modules_df_degree, s_jobs, unlocks_graph, lambda_test)
            
            # BUGFIX: Don't convert to string - core_elective_map has integer/original dtype keys
            core_elective_map = csv_df[csv_df["course"] == course_name].set_index('code')['type'].to_dict()
            
            degree_record = {
                'university': university.upper(),
                'degree': course_name,
                'avg_s_jobs': np.mean([s_jobs.get(str(c), 0.0) for c in modules_df_degree['code']]),
                'avg_s_level': np.mean([extract_module_level(c) / 4000.0 for c in modules_df_degree['code']]),
                'core_ratio': np.mean([1.0 if core_elective_map.get(c, 'elective') == 'core' else 0.0 
                                      for c in modules_df_degree['code']]),
                'num_modules': len(modules_df_degree)
            }
            
            if university == 'nus':
                degree_record['avg_s_prereqs'] = np.mean([s_prereqs.get(str(c), 0.0) for c in modules_df_degree['code']])
            
            degree_features.append(degree_record)
        
        if len(degree_features) == 0:
            continue
        
        features_df = pd.DataFrame(degree_features)
        merged_df = features_df.merge(ges_df, on=['university', 'degree'], how='inner')
        
        if len(merged_df) < 3:
            print(f"Only {len(merged_df)} degrees matched")
            continue
        
        # Prepare features
        if university == 'nus':
            X = merged_df[['avg_s_jobs', 'avg_s_prereqs', 'avg_s_level', 'core_ratio']].values
            feature_names = ['s_jobs', 's_prereqs', 's_level', 'core_boost']
        else:
            X = merged_df[['avg_s_jobs', 'avg_s_level', 'core_ratio']].values
            feature_names = ['s_jobs', 's_level', 'core_boost']
        
        y_employment = merged_df['full_time_permanent_pct'].values
        
        # Standardize and fit OLS
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        model = LinearRegression()
        model.fit(X_scaled, y_employment)
        r2 = model.score(X_scaled, y_employment)
        
        if university == 'nus':
            print(f"R² = {r2:.4f}")
        
        # Track best
        if r2 > best_r2:
            best_r2 = r2
            best_lambda = lambda_test
            best_model = model
            best_merged_df = merged_df
            best_feature_names = feature_names
        
        if university != 'nus':
            break
    
    # Report results
    if best_model is not None:
        print(f"\n✓ Best R² = {best_r2:.4f} ({len(best_merged_df)} degrees)")
        if university == 'nus':
            print(f"✓ Optimal λ = {best_lambda:.1f}")
        
        print("\nLearned weights:")
        for name, coef in zip(best_feature_names, best_model.coef_):
            print(f"  {name:12s}: {coef:+.4f}")
        
        # Save results
        uni_results = {
            'university': university.upper(),
            'lambda': best_lambda if university == 'nus' else None,
            'r2_score': best_r2,
            'intercept': best_model.intercept_,
            'n_degrees': len(best_merged_df)
        }
        for name, coef in zip(best_feature_names, best_model.coef_):
            uni_results[f'coef_{name}'] = coef
        
        all_ols_results.append(uni_results)
        all_merged_dfs.append(best_merged_df)

# Save OLS results
if len(all_ols_results) > 0:
    ols_df = pd.DataFrame(all_ols_results)
    ols_df.to_csv(os.path.join(BASE_DATA_PATH, "ols_learned_weights_by_university.csv"), index=False)
    
    # ADDED: Save combined degree features with employment data for cell 12
    combined_features = pd.concat(all_merged_dfs, ignore_index=True)
    combined_features.to_csv(os.path.join(BASE_DATA_PATH, "degree_features_with_employment.csv"), index=False)
    
    print(f"\n{'='*80}")
    print("✓ Saved: ols_learned_weights_by_university.csv")
    print("✓ Saved: degree_features_with_employment.csv")
    print(f"{'='*80}")
else:
    print("\n⚠ No results to save")

print("\n✓ OLS complete")

UNIVERSITY-SPECIFIC OLS WITH LAMBDA GRID SEARCH

NUS
  Testing λ = 0.0... R² = 0.8680
  Testing λ = 0.1... R² = 0.8727
  Testing λ = 0.2... R² = 0.8727
  Testing λ = 0.3... R² = 0.8727
  Testing λ = 0.4... R² = 0.8727
  Testing λ = 0.5... R² = 0.8727
  Testing λ = 0.6... R² = 0.8727
  Testing λ = 0.7... R² = 0.8727
  Testing λ = 0.8... R² = 0.8727
  Testing λ = 0.9... R² = 0.8727

✓ Best R² = 0.8727 (10 degrees)
✓ Optimal λ = 0.5

Learned weights:
  s_jobs      : +10.7858
  s_prereqs   : +1.5022
  s_level     : +13.0161
  core_boost  : +3.0922

SMU

✓ Best R² = 0.5848 (6 degrees)

Learned weights:
  s_jobs      : +1.8101
  s_level     : -4.2400
  core_boost  : +0.7264

SUTD

✓ Best R² = 0.7867 (5 degrees)

Learned weights:
  s_jobs      : -7.8087
  s_level     : -2.4776
  core_boost  : +0.4942

✓ Saved: ols_learned_weights_by_university.csv
✓ Saved: degree_features_with_employment.csv

✓ OLS complete


In [14]:
# === ALTERNATIVE VALIDATION: SPEARMAN CORRELATION & PERMUTATION TESTS ===
# Better for small samples (SMU: 6 degrees, SUTD: 5 degrees)

from scipy.stats import spearmanr, pearsonr
import matplotlib.pyplot as plt

print("="*80)
print("SPEARMAN CORRELATION ANALYSIS (Better for small samples)")
print("="*80)

# Load degree features
features_df = pd.read_csv(os.path.join(BASE_DATA_PATH, "degree_features_with_employment.csv"))

for university in ['NUS', 'SMU', 'SUTD']:
    uni_data = features_df[features_df['university'] == university].copy()
    
    if len(uni_data) == 0:
        continue
    
    print(f"\n{'='*80}")
    print(f"{university} ({len(uni_data)} degrees)")
    print(f"{'='*80}")
    
    # Features to test
    if university == 'NUS':
        feature_cols = ['avg_s_jobs', 'avg_s_prereqs', 'avg_s_level', 'core_ratio']
    else:
        feature_cols = ['avg_s_jobs', 'avg_s_level', 'core_ratio']
    
    employment = uni_data['full_time_permanent_pct'].values
    
    print("\nSpearman Correlations with Employment:")
    print("-"*80)
    
    correlations = []
    for feat in feature_cols:
        values = uni_data[feat].values
        
        # Check for zero variance (all values the same)
        if np.std(values) == 0:
            print(f"  {feat:20s}: ⚠ Zero variance (all values are {values[0]:.3f})")
            continue
        
        # Spearman correlation (rank-based, robust for small samples)
        rho, p_value = spearmanr(values, employment)
        
        # Pearson for comparison
        r, _ = pearsonr(values, employment)
        
        correlations.append({
            'feature': feat,
            'spearman_rho': rho,
            'p_value': p_value,
            'pearson_r': r,
            'significant': '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else ''
        })
        
        print(f"  {feat:20s}: ρ={rho:+.3f}  p={p_value:.3f}  {correlations[-1]['significant']}")
    
    # Interpretation
    print("\nInterpretation:")
    print("-"*80)
    
    if len(correlations) == 0:
        print(f"⚠ No features with variance to test")
        continue
    
    significant = [c for c in correlations if c['p_value'] < 0.05]
    
    if len(significant) > 0:
        print(f"✓ {len(significant)}/{len(correlations)} features significantly correlate with employment")
        for c in sorted(significant, key=lambda x: abs(x['spearman_rho']), reverse=True):
            direction = "increases" if c['spearman_rho'] > 0 else "decreases"
            print(f"  • {c['feature']}: {direction} employment (ρ={c['spearman_rho']:+.3f}, p={c['p_value']:.3f})")
    else:
        print(f"⚠ No features significantly correlate with employment (p<0.05)")
        print(f"   Sample size ({len(uni_data)} degrees) may be too small for statistical power")
    
    # Permutation test for overall predictive power
    print(f"\nPermutation Test:")
    print("-"*80)
    
    # Compute actual correlation of best feature
    best_feat = max(correlations, key=lambda x: abs(x['spearman_rho']))
    actual_corr = abs(best_feat['spearman_rho'])
    
    # Shuffle employment 1000 times
    np.random.seed(42)
    null_distribution = []
    for _ in range(1000):
        shuffled_employment = np.random.permutation(employment)
        rho, _ = spearmanr(uni_data[best_feat['feature']].values, shuffled_employment)
        null_distribution.append(abs(rho))
    
    # P-value: fraction of shuffled correlations >= actual
    perm_p_value = np.mean(np.array(null_distribution) >= actual_corr)
    
    print(f"Best feature: {best_feat['feature']} (|ρ|={actual_corr:.3f})")
    print(f"Permutation p-value: {perm_p_value:.3f}")
    
    if perm_p_value < 0.05:
        print(f"✓ Features are better than random (p={perm_p_value:.3f})")
    else:
        print(f"⚠ Features not significantly better than random (p={perm_p_value:.3f})")

print("\n" + "="*80)
print("VALIDATION SUMMARY")
print("="*80)
print("• Spearman correlation: Non-parametric, works with small samples")
print("• P-value < 0.05: Statistically significant relationship")
print("• ***: p<0.001, **: p<0.01, *: p<0.05")
print("• Permutation test: Checks if features beat random chance")
print("="*80)

SPEARMAN CORRELATION ANALYSIS (Better for small samples)

NUS (10 degrees)

Spearman Correlations with Employment:
--------------------------------------------------------------------------------
  avg_s_jobs          : ρ=+0.685  p=0.029  *
  avg_s_prereqs       : ρ=+0.673  p=0.033  *
  avg_s_level         : ρ=+0.612  p=0.060  
  core_ratio          : ρ=-0.479  p=0.162  

Interpretation:
--------------------------------------------------------------------------------
✓ 2/4 features significantly correlate with employment
  • avg_s_jobs: increases employment (ρ=+0.685, p=0.029)
  • avg_s_prereqs: increases employment (ρ=+0.673, p=0.033)

Permutation Test:
--------------------------------------------------------------------------------
Best feature: avg_s_jobs (|ρ|=0.685)
Permutation p-value: 0.029
✓ Features are better than random (p=0.029)

SMU (6 degrees)

Spearman Correlations with Employment:
--------------------------------------------------------------------------------
  avg_s_jo